In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,PDFPlumberLoader
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_groq import ChatGroq
from tqdm.notebook import tqdm

In [2]:
pdf_path = r'docs\Lenovo_user_guide.pdf'

In [3]:
loader = PyMuPDFLoader(pdf_path)
doc = loader.load()

c:\Users\Rajeev\.conda\envs\llama_env\lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:322: UserWarning: Warning: Empty content on page 174 of document docs\Lenovo_user_guide.pdf
  warnings.warn(
c:\Users\Rajeev\.conda\envs\llama_env\lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:322: UserWarning: Warning: Empty content on page 175 of document docs\Lenovo_user_guide.pdf
  warnings.warn(
c:\Users\Rajeev\.conda\envs\llama_env\lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:322: UserWarning: Warning: Empty content on page 176 of document docs\Lenovo_user_guide.pdf
  warnings.warn(


In [4]:
doc[6].page_content

'Read this first\nBe sure to follow the important tips given here to get the most use and enjoyment out of your computer.\nFailure to do so might lead to discomfort or injury, or cause your computer to fail.\nProtect yourself from the heat that your computer generates.\nWhen your computer is turned on or the battery is charging, the base, the palm\nrest, and some other parts may become hot. The temperature they reach\ndepends on the amount of system activity and the level of charge in the battery.\nExtended contact with your body, even through clothing, could cause\ndiscomfort or even a skin burn.\n• Avoid keeping your hands, your lap, or any other part of your body in\ncontact with a hot section of the computer for any extended time.\n• Periodically take hands from using the keyboard by lifting your hands from\nthe palm rest.\nProtect yourself from the heat generated by the ac power adapter.\nWhen the ac power adapter is connected to an electrical outlet and your\ncomputer, it generat

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                 chunk_overlap=200)

doc_split = splitter.split_documents(doc)

In [6]:
model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model_kwargs = {'device':'cpu'}

In [7]:
hf = HuggingFaceEmbeddings(model_name=model_name,
                           model_kwargs=model_kwargs)

In [8]:
embeddings = hf.embed_documents([doc.page_content for doc in tqdm(doc_split)])

  0%|          | 0/470 [00:00<?, ?it/s]

In [9]:
from uuid import uuid4

[str(uuid4()) for _ in range(5)]

['66f93e83-507b-4a3c-a5bc-ac57205e7ede',
 '5b3cb077-a9ca-4d3d-b624-13da63d43212',
 'cd24cc8b-4fca-43a3-93fc-a0e8351524eb',
 'bb067371-5b30-4054-afba-fd3f2ff32d29',
 '0f6cc24d-e8cc-4dec-8459-5227a7a3566c']

In [10]:
import faiss

index = faiss.IndexFlatL2(len(hf.embed_query("hello world")))

In [11]:
vector_store = FAISS(embedding_function=hf,
                     index=index,
                     docstore=InMemoryDocstore(),
                     index_to_docstore_id={})

In [12]:
from uuid import uuid4

docs = [doc.page_content for doc in doc_split]

uuids = [str(uuid4()) for _ in range(len(doc_split))]
vector_store.add_documents(documents=doc_split, ids=uuids)

['1499b344-af14-4cd8-9d12-844832656762',
 'd03b315f-bf74-4db9-9d46-c767a188c8dc',
 '0d0b1d31-0f30-4534-ad4c-d42e5d1a5534',
 '55b0eac9-84d6-4f65-8424-4c6e148d77c1',
 'a580e47f-e400-42f9-b1eb-9ddf2ecdca7b',
 'e48d20be-8463-4875-9ee8-96497a14b329',
 '9647d810-0c88-484e-8e0a-dc65cdd66327',
 'e8410932-0fa3-4623-8e8c-a0fde0c869ca',
 'f9a4fa47-7717-48fd-9494-15e65485a0a9',
 '0539d7c4-0971-4e8d-bd53-1915e6250632',
 '79c93c97-4451-428d-9ba8-6a18ce5db4a1',
 '812dcf25-d31a-440d-925b-870da821c5b3',
 'abf2a001-03f6-4b27-b976-e0b513567060',
 '662aa36c-4fbb-4826-b4cc-7df3e1ac5515',
 'f55016b7-f2c3-4eed-8a2d-42c9b4566d04',
 '0c3289d2-a63f-4422-a6df-e33290438f0b',
 '5653b9c4-e28f-4ba5-80be-efa1ea248b81',
 '5534ffb5-114c-4b25-b61f-0d0d3a0f65a0',
 '553187f9-21fb-4d79-8e63-2bb511c3f543',
 'a77c11f8-434f-4cb8-b8ad-aea47b5ed8cc',
 '68a5639b-5707-4acb-be52-60830ccd87db',
 '8f0c3674-4b59-4645-b863-40ca3d9cdae0',
 'f8469b48-49eb-4431-a880-2e8014bc9865',
 '3ce8c78d-bc33-4490-a47d-d63e175c4947',
 'a16a3b63-7075-

In [13]:
# results = vector_store.similarity_search_with_score(
#     "What re the media cards which are supported?",
#     k=3)

# for res,score in results:
#     print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

In [14]:
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory.buffer import ConversationBufferMemory
from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain,
    ConversationalRetrievalChain,
    )
from langchain.chains.combine_documents import create_stuff_documents_chain

In [15]:
memory = ConversationBufferMemory(memory_key='chat_history',
                                  return_messages=True)
retriever = vector_store.as_retriever()

C:\Users\Rajeev\AppData\Local\Temp\ipykernel_22996\2712487620.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history',


In [16]:
llm = ChatGroq(model_name="llama3-8b-8192",)

In [17]:
# Contextualize question
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, just "
    "reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# Answer question
qa_system_prompt = (
    "You are an assistant for question-answering tasks. Use "
    "the following pieces of retrieved context to answer the "
    "question. If you don't know the answer, just say that you "
    "don't know. Keep the answer to the point."
    "{context}"
)

In [18]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [19]:
qa_chain = create_stuff_documents_chain(llm,qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

In [20]:
chat_history = memory.load_memory_variables({})['chat_history']

In [28]:
chat_history = memory.load_memory_variables({})['chat_history']
print(chat_history)

query = "I mean computer virus?"

response = rag_chain.invoke({
    "chat_history":chat_history,
    "input":query})
print(response["answer"])

memory.save_context(
        {"input": query},
        {"output": response["answer"]}
    )

[HumanMessage(content='How can I start the camera?', additional_kwargs={}, response_metadata={}), AIMessage(content='To start the camera, follow these steps:\n\n* For Windows 7: Start the Communications Utility program.\n* For Windows 8.1: Click Camera from the Start screen.\n* For Windows 10: Open the Start menu, and click Camera from the all apps list.\n\nWhen the camera is started, the green camera-in-use indicator will turn on. You can also use the integrated camera with other programs that provide features such as photographing, video capturing, and video conferencing.', additional_kwargs={}, response_metadata={}), HumanMessage(content='How can I configure its settings?', additional_kwargs={}, response_metadata={}), AIMessage(content='To configure the camera settings, follow these steps:\n\n* For Windows 7: Start the Communications Utility program and configure the camera settings as desired.\n* For Windows 8.1:\n\t+ Configure the camera settings directly from the program that is 

In [22]:
# memory.clear()